## task 1

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

data = pd.read_csv("heart.csv")
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,Male,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,Male,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,Male,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,Male,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,Female,0,138,294,1,1,106,0,1.9,1,3,2,0


In [2]:
data.shape, data.isnull().sum()

((1025, 14),
 age         0
 sex         0
 cp          0
 trestbps    0
 chol        0
 fbs         0
 restecg     0
 thalach     0
 exang       0
 oldpeak     0
 slope       0
 ca          0
 thal        0
 target      0
 dtype: int64)

In [3]:
feature_columns = [column for column in data.columns if column != "target"]
numeric_ranges = {
    "age": (18, 100),
    "trestbps": (60, 250),
    "chol": (80, 700),
    "thalach": (40, 230),
    "oldpeak": (-5, 15),
    "ca": (0, 4)
}
category_values = {
    "sex": ["Male", "Female"],
    "cp": [0, 1, 2, 3],
    "fbs": [0, 1],
    "restecg": [0, 1, 2],
    "exang": [0, 1],
    "slope": [0, 1, 2],
    "thal": [0, 1, 2, 3]
}

In [4]:
def suggest_value(column, value):
    if column in numeric_ranges:
        low, high = numeric_ranges[column]
        try:
            return min(max(float(value), low), high)
        except (TypeError, ValueError):
            return data[column].median()
    if column in category_values:
        return data[column].mode().iloc[0]
    return value

In [5]:
def validate_input(values):
    cleaned, errors, suggestions = {}, {}, {}
    for column in feature_columns:
        value = values.get(column)
        if value is None or str(value).strip() == "":
            errors[column] = "This field is required."
            suggestions[column] = suggest_value(column, value)
            continue
        if column in numeric_ranges:
            try:
                value = float(value)
                low, high = numeric_ranges[column]
                if not low <= value <= high:
                    errors[column] = f"Enter a value between {low} and {high}."
                    suggestions[column] = suggest_value(column, value)
                    continue
                if column != "oldpeak":
                    value = int(value)
            except (TypeError, ValueError):
                errors[column] = "Enter a valid number."
                suggestions[column] = data[column].median()
                continue
        if column in category_values and value not in category_values[column]:
            errors[column] = f"Choose one of: {category_values[column]}."
            suggestions[column] = data[column].mode().iloc[0]
            continue
        cleaned[column] = value
    return cleaned, errors, suggestions

In [6]:
sample_input = data[feature_columns].iloc[0].to_dict()
sample_input["age"] = 150
sample_input["chol"] = "wrong"
cleaned_input, input_errors, input_suggestions = validate_input(sample_input)
input_errors, input_suggestions

({'age': 'Enter a value between 18 and 100.', 'chol': 'Enter a valid number.'},
 {'age': 100, 'chol': np.float64(240.0)})

In [7]:
for column, message in input_errors.items():
    print(column, ":", message, "| Suggested value:", input_suggestions[column])

age : Enter a value between 18 and 100. | Suggested value: 100
chol : Enter a valid number. | Suggested value: 240.0


## task 2

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = data[feature_columns].copy()
y = data["target"].astype(int)
categorical_columns = ["sex"]
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced"))
])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(classification_report(y_test, predictions))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205



In [10]:
def prediction_confidence(probabilities):
    confidence = max(probabilities)
    if confidence >= 0.80:
        return "high"
    if confidence >= 0.60:
        return "moderate"
    return "low"

def explain_prediction(values, fitted_model=model):
    row = pd.DataFrame([values], columns=feature_columns)
    probabilities = fitted_model.predict_proba(row)[0]
    predicted_class = int(fitted_model.predict(row)[0])
    confidence = float(probabilities[predicted_class])
    base_probability = float(probabilities[1])
    effects = []

    for column in feature_columns:
        changed = row.copy()
        changed[column] = data[column].median() if column in numeric_columns else data[column].mode().iloc[0]
        changed_probability = float(fitted_model.predict_proba(changed)[0][1])
        effects.append((column, base_probability - changed_probability))

    effects.sort(key=lambda item: abs(item[1]), reverse=True)
    important_effects = effects[:4]
    result_text = (
        "The model predicts the positive heart-disease class."
        if predicted_class == 1 else
        "The model predicts the negative heart-disease class."
    )
    details = []
    for column, effect in important_effects:
        direction = "higher" if effect > 0 else "lower"
        details.append(f"{column} was linked to a {direction} model probability compared with its typical value")

    level = prediction_confidence(probabilities)
    explanation = result_text + " " + "; ".join(details) + "."
    if level == "high":
        explanation += " Confidence in this predicted class is high."
    elif level == "moderate":
        explanation += " Confidence is moderate, so interpret the result cautiously."
    else:
        explanation += " Confidence is low and the prediction is uncertain."
    explanation += " This is a machine-learning estimate, not a medical diagnosis."

    return {
        "prediction": predicted_class,
        "probabilities": probabilities,
        "confidence": confidence,
        "confidence_level": level,
        "explanation": explanation,
        "feature_effects": important_effects
    }

In [11]:
example_result = explain_prediction(X_test.iloc[0].to_dict())
print(example_result["explanation"])
print("Predicted class:", example_result["prediction"])
print("Confidence:", round(example_result["confidence"] * 100, 2), "%")

The model predicts the negative heart-disease class. ca was linked to a lower model probability compared with its typical value; cp was linked to a lower model probability compared with its typical value; thalach was linked to a lower model probability compared with its typical value; restecg was linked to a lower model probability compared with its typical value. Confidence in this predicted class is high. This is a machine-learning estimate, not a medical diagnosis.
Predicted class: 0
Confidence: 98.8 %


## task 3

In [12]:
import joblib
from pathlib import Path

joblib.dump(model, "heart_model.pkl")
joblib.dump(feature_columns, "heart_features.pkl")

['heart_features.pkl']

In [13]:
app_code = '\nimport streamlit as st\nimport pandas as pd\nimport joblib\n\nst.set_page_config(page_title="Heart Prediction", page_icon="❤️", layout="wide")\nmodel = joblib.load("heart_model.pkl")\nfeature_columns = joblib.load("heart_features.pkl")\n\nnumeric_ranges = {\n    "age": (18, 100), "trestbps": (60, 250), "chol": (80, 700),\n    "thalach": (40, 230), "oldpeak": (-5, 15), "ca": (0, 4)\n}\ncategory_values = {\n    "sex": ["Male", "Female"], "cp": [0, 1, 2, 3], "fbs": [0, 1],\n    "restecg": [0, 1, 2], "exang": [0, 1], "slope": [0, 1, 2],\n    "thal": [0, 1, 2, 3]\n}\n\nst.markdown("""\n<style>\n.stApp {\n    background: linear-gradient(135deg, #0b132b 0%, #1c2541 55%, #243b55 100%);\n    color: #f8fafc;\n}\n[data-testid="stHeader"] { background: transparent; }\n[data-testid="stSidebar"] { background: #101b33; border-right: 1px solid #425477; }\nh1, h2, h3, p, label, [data-testid="stMarkdownContainer"] { color: #f8fafc !important; }\ndiv[data-testid="stForm"] {\n    background: rgba(255,255,255,0.07);\n    padding: 22px;\n    border: 1px solid rgba(255,255,255,0.18);\n    border-radius: 18px;\n}\n.stButton > button {\n    background: linear-gradient(90deg, #00b4d8, #48cae4);\n    color: #071526; border: 0; border-radius: 10px;\n    font-weight: 700; transition: all .25s ease;\n}\n.stButton > button:hover {\n    transform: translateY(-2px);\n    box-shadow: 0 8px 22px rgba(0,180,216,.30);\n}\n[data-testid="stMetric"] {\n    background: rgba(255,255,255,.08); padding: 16px;\n    border-radius: 14px; border: 1px solid rgba(255,255,255,.15);\n}\n.stAlert { border-radius: 12px; }\ninput, textarea, [data-baseweb="select"] { border-radius: 9px !important; }\n@media (max-width: 700px) { .block-container { padding: 1rem; } }\n</style>\n""", unsafe_allow_html=True)\n\nst.title("❤️ Heart Health Prediction")\nst.caption("Enter the health measurements to view a machine-learning estimate.")\n\nwith st.form("heart_input_form"):\n    left, right = st.columns(2)\n    values = {}\n    with left:\n        values["age"] = st.number_input("Age", min_value=0, max_value=120, value=45)\n        values["sex"] = st.selectbox("Sex", category_values["sex"])\n        values["cp"] = st.selectbox("Chest pain type (cp)", category_values["cp"])\n        values["trestbps"] = st.number_input("Resting blood pressure", min_value=0, max_value=300, value=120)\n        values["chol"] = st.number_input("Cholesterol", min_value=0, max_value=800, value=200)\n        values["fbs"] = st.selectbox("Fasting blood sugar (fbs)", category_values["fbs"])\n        values["restecg"] = st.selectbox("Resting ECG (restecg)", category_values["restecg"])\n    with right:\n        values["thalach"] = st.number_input("Maximum heart rate (thalach)", min_value=0, max_value=250, value=150)\n        values["exang"] = st.selectbox("Exercise-induced angina (exang)", category_values["exang"])\n        values["oldpeak"] = st.number_input("Oldpeak", min_value=-10.0, max_value=20.0, value=1.0, step=0.1)\n        values["slope"] = st.selectbox("Slope", category_values["slope"])\n        values["ca"] = st.number_input("Number of major vessels (ca)", min_value=0, max_value=6, value=0)\n        values["thal"] = st.selectbox("Thal", category_values["thal"])\n    submitted = st.form_submit_button("Predict")\n\ndef validate(values):\n    errors, suggestions, cleaned = {}, {}, {}\n    for column in feature_columns:\n        value = values[column]\n        if column in numeric_ranges:\n            low, high = numeric_ranges[column]\n            if not low <= value <= high:\n                errors[column] = f"Allowed range: {low} to {high}."\n                suggestions[column] = min(max(value, low), high)\n                continue\n        cleaned[column] = value\n    return cleaned, errors, suggestions\n\ndef confidence_label(value):\n    if value >= 0.80:\n        return "High"\n    if value >= 0.60:\n        return "Moderate"\n    return "Low"\n\nif submitted:\n    cleaned, errors, suggestions = validate(values)\n    if errors:\n        st.error("Please correct the following inputs.")\n        for field, message in errors.items():\n            st.warning(f"{field}: {message} Suggested value: {suggestions[field]}")\n    else:\n        row = pd.DataFrame([cleaned], columns=feature_columns)\n        probabilities = model.predict_proba(row)[0]\n        prediction = int(model.predict(row)[0])\n        confidence = float(probabilities[prediction])\n        level = confidence_label(confidence)\n\n        if prediction == 1:\n            st.warning("The model predicts the positive heart-disease class.")\n        else:\n            st.success("The model predicts the negative heart-disease class.")\n\n        first, second = st.columns(2)\n        first.metric("Prediction confidence", f"{confidence * 100:.1f}%")\n        second.metric("Confidence level", level)\n        st.progress(confidence)\n\n        if confidence >= 0.80:\n            confidence_note = "The model is relatively confident in this result."\n        elif confidence >= 0.60:\n            confidence_note = "The model has moderate confidence. Interpret the result carefully."\n        else:\n            confidence_note = "The model is uncertain. Avoid relying on this prediction."\n\n        st.subheader("What this prediction means")\n        st.write(\n            ("The model predicts the positive heart-disease class. " if prediction == 1\n             else "The model predicts the negative heart-disease class.")\n            + confidence_note\n        )\n        st.info("This tool provides an educational machine-learning estimate and is not a medical diagnosis. Consult a qualified healthcare professional for medical advice.")\n'
Path('heart_app.py').write_text(app_code, encoding='utf-8')
print('Streamlit app file created: heart_app.py')

Streamlit app file created: heart_app.py


In [14]:
print("Run this command in a terminal:")
print("streamlit run heart_app.py")

Run this command in a terminal:
streamlit run heart_app.py
